In [178]:
import os
import pandas as pd
import re
from bisect import bisect_right

# ----------------------------
# SETTINGS
# ----------------------------
WIKI_TREATIES = "./wiki-treaties_formatted.csv"
UNO_TREATIES = "./UNO-Treaties.csv"
INSTRUMENTS = "../instruments/ohchr_instruments_detailed.csv"
CONV_PROT_REC = "../conv-prot-rec/conventions-protocols-recommendations.csv"
RESOLUTIONS_TRIM = "../resolutions/ga_resolutions_1946_2019_3100-trim.csv"

df_wiki = pd.read_csv(WIKI_TREATIES)
df_uno = pd.read_csv(UNO_TREATIES)
df_ins = pd.read_csv(INSTRUMENTS)
df_conv = pd.read_csv(CONV_PROT_REC)
df_res = pd.read_csv(RESOLUTIONS_TRIM)

df_res = df_res.head(100)

df_wiki.head()

,year,name,url,note,summary,cleaned_note,cleaned_title
0,1900,Treaty of Paris,https://en.wikipedia.org/wiki/Treaty_of_Paris_...,NaN,Ends all conflicting claims over Río Muni ( Eq...,NaN,Treaty of Paris
1,1900,Treaty of Washington,https://en.wikipedia.org/wiki/Treaty_of_Washin...,NaN,Seeks to remove any ground of misunderstanding...,NaN,Treaty of Washington
2,1900,Convention for the Preservation of Wild Animal...,https://en.wikipedia.org/wiki/Convention_for_t...,NaN,First international agreement on wildlife cons...,NaN,Convention for the Preservation of Wild Animal...
3,1901,Hay–Pauncefote Treaty,https://en.wikipedia.org/wiki/Hay%E2%80%93Paun...,NaN,Replaces the Clayton–Bulwer Treaty,NaN,Hay–Pauncefote Treaty
4,1901,Boxer Protocol,https://en.wikipedia.org/wiki/Boxer_Protocol,Also known as the Peace Agreement between the ...,Peace agreement between the Eight-Nation Allia...,Peace Agreement between the Great Powers and C...,Boxer Protocol


In [179]:
print(f"WIKI_TREATIES columns: {df_wiki.columns}")
print(f"UNO_TREATIES columns: {df_uno.columns}")
print(f"INSTRUMENTS columns: {df_ins.columns}")
print(f"CONV_PROT_REC columns: {df_conv.columns}")
print(f"RESOLUTIONS_TRIM columns: {df_res.columns}")

WIKI_TREATIES columns: Index(['year', 'name', 'url', 'note', 'summary', 'cleaned_note',
       'cleaned_title'],
      dtype='object')
UNO_TREATIES columns: Index(['title', 'location', 'date', 'chapter'], dtype='object')
INSTRUMENTS columns: Index(['title', 'url', 'adopted_by', 'content', 'pdf_url'], dtype='object')
CONV_PROT_REC columns: Index(['code', 'title', 'year', 'number'], dtype='object')
RESOLUTIONS_TRIM columns: Index(['res_id2', 'part', 'res_id3', 'res_id2_unlet', 'alt_id_dic',
       'session_type', 'session_reg', 'session_sp', 'session_es', 'resn',
       'res_letter', 'date_p', 'date_c', 'filename', 'content', 'location',
       'record', 'draft', 'topic', 'n_inc_cit'],
      dtype='object')


In [180]:
print(f"{'WIKI_TREATIES':=^30}")
print(f"Number of unique cleaned titles: {df_wiki['cleaned_title'].nunique()}\n\
        Length of all titles: {len(df_wiki['cleaned_title'])}")
print(f"Number of unique cleaned alternative names: {df_wiki['cleaned_note'].nunique()}\n\
        Length of all alternative names: {len(df_wiki['cleaned_note'].dropna())}")


print("Non-unique cleaned titles:")
non_unique_titles = df_wiki[df_wiki.duplicated(subset='cleaned_title', keep=False)]
non_unique_titles[['year', 'cleaned_title', 'name']]

========WIKI_TREATIES=========
Number of unique cleaned titles: 345
        Length of all titles: 364
Number of unique cleaned alternative names: 101
        Length of all alternative names: 102
Non-unique cleaned titles:


,year,cleaned_title,name
21,1905,Japan–Korea Treaty,Japan–Korea Treaty of 1905
28,1910,Japan–Korea Treaty,Japan–Korea Treaty of 1910
34,1913,Treaty of London,Treaty of London (1913)
35,1913,Treaty of Bucharest,Treaty of Bucharest (1913)
40,1915,Treaty of London,Treaty of London (1915) (London Pact)
44,1916,Treaty of Bucharest,Treaty of Bucharest (1916)
51,1918,Treaty of Bucharest,Treaty of Bucharest (1918)
64,1920,Treaty of Warsaw,Treaty of Warsaw (1920)
66,1920,Treaty of Rapallo,Treaty of Rapallo (1920)
67,1920,Treaty of Moscow,Treaty of Moscow (1920)


In [181]:
print(f"{'UNO_TREATIES':=^50}")
print(f"Number of unique titles: {df_uno['title'].nunique()}\n\
        Length of all titles: {len(df_uno['title'])}")

dupes = (
    df_uno.groupby(["title", "location"])
          .size()
          .reset_index(name="count")
          .query("count > 1")
)
print('Duplicates considering the tile and the loc:')
print(dupes, "\n\n\n")

print(f"{'INSTRUMENTS':=^50}")
print(f"Number of unique titles: {df_ins['title'].nunique()}\n\
        Length of all titles: {len(df_ins['title'])}")

dupes = (
    df_ins.groupby(["title", "adopted_by"])
          .size()
          .reset_index(name="count")
          .query("count > 1")
)

print('Duplicates considering the tile and the date:')
print(dupes, "\n\n\n")

print(f"{'ILO CONVENTIONS AND PROTOCOLS':=^50}")
print(f"Number of unique titles: {df_conv['title'].nunique()}\n\
        Length of all titles: {len(df_conv['title'])}")

dupes = (
    df_conv.groupby(["title", "year"])
          .size()
          .reset_index(name="count")
          .query("count > 1")
)

print('Duplicates considering the tile and the date:')
print(dupes)


===================UNO_TREATIES===================
Number of unique titles: 398
        Length of all titles: 428
Duplicates considering the tile and the loc:
                                                 title  location  count
74   Amendments to the Convention on the Internatio...    London      2
82   Amendments to the title and substantive provis...    London      2
185  Customs Convention on the International Transp...    Geneva      2
194  European Agreement concerning the Work of Crew...    Geneva      2
231                      International Cocoa Agreement    Geneva      5
234                     International Coffee Agreement    London      2
235                     International Coffee Agreement  New York      3
269             International Natural Rubber Agreement    Geneva      2
274                      International Sugar Agreement    Geneva      4
276                      International Sugar Agreement  New York      2 



===================INSTRUMENTS===============

In [182]:
'''
De los titulos que si son unicos, podemos usarlos directamente, sin especificar el año.
De los titulos que no son unicos, tenemos que usar el año. Podemos usarlo de las siguientes maneras
- El titulo seguido de "of YEAR"
- El titulo seguido de (YEAR)
- El titulo precedido por YEAR
- El titulo precedido por (YEAR)
Solo el titulo, extraer el fragmento con un padding de 10 caracteres y decidir
'''

'\nDe los titulos que si son unicos, podemos usarlos directamente, sin especificar el año.\nDe los titulos que no son unicos, tenemos que usar el año. Podemos usarlo de las siguientes maneras\n- El titulo seguido de "of YEAR"\n- El titulo seguido de (YEAR)\n- El titulo precedido por YEAR\n- El titulo precedido por (YEAR)\nSolo el titulo, extraer el fragmento con un padding de 10 caracteres y decidir\n'

In [183]:
# Lowercase, otherwise nothing is detected
df_wiki["cleaned_title"] = df_wiki["cleaned_title"].str.lower()

# -------------------------
# Separate unique/non-unique
# -------------------------
unique_titles = df_wiki.drop_duplicates("cleaned_title", keep=False)

non_unique_titles = df_wiki[
    df_wiki.duplicated("cleaned_title", keep=False)
]

# -------------------------
# Collect matches
# -------------------------
MAX_WORD_DIST = 6

records = []

# --- 1. Build map: pattern -> (treaty_id, treaty_title) ---
all_known = []  # (pattern, treaty_id, title)

# Unique titles (no year needed at detection)
for _, t in unique_titles.iterrows():
    all_known.append((
        re.escape(t["cleaned_title"]),
        f'{t["cleaned_title"]};{t["year"]}',
        t["cleaned_title"],
    ))

# Non-unique titles (allow year within MAX_WORD_DIST words)
for _, t in non_unique_titles.iterrows():
    year = str(int(t["year"]))
    title = t["cleaned_title"]

    pat = (
        rf"\b{re.escape(title)}\b(?:\W+\w+){{0,{MAX_WORD_DIST}}}\W+{year}\b"
        rf"|"
        rf"\b{year}\b(?:\W+\w+){{0,{MAX_WORD_DIST}}}\W+{re.escape(title)}\b"
        rf"|"
        rf"\b{re.escape(title)}\s*\({year}\)"
        rf"|"
        rf"\({year}\)\s*{re.escape(title)}"
        rf"|"
        rf"\b{re.escape(title)}\s+of\s+{year}\b"
    )

    all_known.append((pat, f"{title};{year}", title))

# repite para df_1, df_2, df_3 cuando los tengas (con su propio identifier/URL)

# --- 2. Compila UN solo regex con grupos nombrados (o usa un dict índice->id) ---
# Con muchos títulos, mejor usar índice numérico como nombre de grupo
combined_pattern = "|".join(f"(?P<t{i}>{p})" for i, (p, _, _) in enumerate(all_known))
combined_re = re.compile(combined_pattern)
id_lookup = {f"t{i}": (tid, title) for i, (_, tid, title) in enumerate(all_known)}


In [184]:
import os
import pandas as pd
import re
from bisect import bisect_right

# ----------------------------
# SETTINGS
# ----------------------------

df_wiki = pd.read_csv(WIKI_TREATIES)
df_uno = pd.read_csv(UNO_TREATIES)
df_ins = pd.read_csv(INSTRUMENTS)
df_conv = pd.read_csv(CONV_PROT_REC)
df_res = pd.read_csv(RESOLUTIONS_TRIM)
df_uno["year"] = pd.to_datetime(df_uno["date"], format="%d %B %Y").dt.year

df_res = df_res.head(100)


MAX_WORD_DIST = 6


# ----------------------------------------------------
# Function to create regex patterns from a dataframe
# ----------------------------------------------------
def build_known_patterns(
    df,
    title_col,
    identifier_func,
    unique_by=None,
    allow_year_disambiguation=False,
    year_col="year"
):
    """
    Returns:
        list of tuples:
        (regex_pattern, identifier, title)
    """

    patterns = []

    # Default: everything is unique by title
    if unique_by is None:
        unique_rows = df
        non_unique_rows = pd.DataFrame(columns=df.columns)
    else:
        unique_rows = df.drop_duplicates(unique_by, keep=False)
        non_unique_rows = df[df.duplicated(unique_by, keep=False)]


    # -----------------------------
    # Unique titles
    # -----------------------------
    for _, row in unique_rows.iterrows():

        title = row[title_col].lower()

        patterns.append((
            rf"\b{re.escape(title)}\b",
            identifier_func(row),
            title
        ))


    # -----------------------------
    # Non unique titles
    # -----------------------------
    if allow_year_disambiguation:

        for _, row in non_unique_rows.iterrows():

            title = row[title_col].lower()
            year = str(int(row[year_col]))

            pat = (
                rf"\b{re.escape(title)}\b(?:\W+\w+){{0,{MAX_WORD_DIST}}}\W+{year}\b"
                rf"|"
                rf"\b{year}\b(?:\W+\w+){{0,{MAX_WORD_DIST}}}\W+{re.escape(title)}\b"
                rf"|"
                rf"\b{re.escape(title)}\s*\({year}\)"
                rf"|"
                rf"\({year}\)\s*{re.escape(title)}"
                rf"|"
                rf"\b{re.escape(title)}\s+of\s+{year}\b"
            )

            patterns.append((
                pat,
                identifier_func(row),
                title
            ))

    return patterns



# ----------------------------------------------------
# Build all known treaty/instrument patterns
# ----------------------------------------------------

all_known = []


# 1. Wikipedia treaties
all_known.extend(
    build_known_patterns(
        df_wiki,
        title_col="cleaned_title",
        unique_by="cleaned_title",
        allow_year_disambiguation=True,
        identifier_func=lambda r: f'{r["cleaned_title"]};{r["year"]}',
        year_col="year"
    )
)


# 2. UNO Treaties
# Titles are not always unique -> Title + Date
all_known.extend(
    build_known_patterns(
        df_uno,
        title_col="title",
        unique_by=["title", "date"],
        allow_year_disambiguation=True,
        identifier_func=lambda r: f'{r["title"]};{r["date"]}',
        year_col="year"
    )
)


# 3. OHCHR Instruments
# URL is unique
all_known.extend(
    build_known_patterns(
        df_ins,
        title_col="title",
        unique_by="title",
        allow_year_disambiguation=False,
        identifier_func=lambda r: r["url"]
    )
)


# 4. Conventions / Protocols / Recommendations
# Code is unique
all_known.extend(
    build_known_patterns(
        df_conv,
        title_col="title",
        unique_by="code",
        allow_year_disambiguation=False,
        identifier_func=lambda r: r["code"]
    )
)



# ----------------------------------------------------
# Compile ONE regex
# ----------------------------------------------------

combined_pattern = "|".join(
    f"(?P<t{i}>{pattern})"
    for i, (pattern, _, _) in enumerate(all_known)
)

combined_re = re.compile(
    combined_pattern,
    flags=re.IGNORECASE
)


id_lookup = {
    f"t{i}": (identifier, title)
    for i, (_, identifier, title) in enumerate(all_known)
}


In [185]:
records = []
candidates = []

KEYWORDS = [
    "agreement",
    "treaty",
    "convention",
    "protocol",
    "amendment",
    "charter",
    "covenant",
    "pact"
]

AVOID_KEYWORDS = [
    r"treaty series",
    r"treaty\s+series,\s*vol\.",
]

keyword_re = re.compile(
    r"\b(" + "|".join(KEYWORDS) + r")\b",
    re.IGNORECASE
)

avoid_re = re.compile(
    "|".join(AVOID_KEYWORDS),
    re.IGNORECASE
)


WINDOW_CHARS = 100

for _, res in df_res.iterrows():
    content = res["content"]
    res_id2 = res["res_id2"]
    res_part = res["part"]

    known_spans = []  # list of (start, end) already matched

    # --- Known treaty matches ---
    for m in combined_re.finditer(content):
        group_name = m.lastgroup
        tid, title = id_lookup[group_name]

        start = m.start()
        end = m.end()
        window = content[max(0, start - WINDOW_CHARS): min(len(content), end + WINDOW_CHARS)]

        records.append({
            "res_id2": res_id2,
            "res_part": res_part,
            "document": title,
            "id": tid,
            "start": start,
            "end": end,
            "context": window
        })

        known_spans.append((start, end))

    known_spans.sort()
    starts = [s for s, e in known_spans]

    # --- Standalone keyword matches not overlapping known treaties ---
    for km in keyword_re.finditer(content):
        k_start, k_end = km.start(), km.end()

        # Ignore "treaty series, vol." references
        nearby = content[max(0, k_start-30): k_end+30]

        if avoid_re.search(nearby):
            continue

        idx = bisect_right(starts, k_start) - 1
        overlapped = False

        for j in (idx, idx + 1):
            if 0 <= j < len(known_spans):
                s, e = known_spans[j]
                if s <= k_start < e or s < k_end <= e:
                    overlapped = True
                    break

        if not overlapped:
            window = content[max(0, k_start - WINDOW_CHARS): min(len(content), k_end + WINDOW_CHARS)]

            candidates.append({
                "res_id2": res_id2,
                "res_part": res_part,
                "keyword": km.group(),
                "start": k_start,
                "end": k_end,
                "context": window
            })

cites = pd.DataFrame(records).drop_duplicates().reset_index(drop=True)
unknown_candidates = pd.DataFrame(candidates).drop_duplicates().reset_index(drop=True)

In [186]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
unknown_candidates

,res_id2,res_part,keyword,start,end,context
0,997 (es-i),1,agreement,287,296,"ces of israel have penetrated deeply into egyptian terri-tory in violation of the general armistice agreement between egypt and israel of 24 february 1949,"" noting that armed forces of france and the united ki"
1,997 (es-i),1,charter,1720,1727,"o the general assembly, for such further action as they may deem appropriate in accordance with the charter; 6. decides to remain in emergency session pending compliance with the present resolution. 562nd pl"
2,1004 (es-ii),1,treaty,270,276,"nt of human rights and of fundamental freedom in hurigary was specifically guar-anteed by the peace treaty between hungary and the allied and associated powers signed at paris on 10 february 1947, and that"
3,1005 (es-ii),1,treaty,950,956,"iet forces in hungary constitutes a violation of the charter of the united nations and of the peace treaty between hungary and the allied and associated powers, considering that the immediate withdrawal of"
4,1237 (es-iii),1,charter,253,260,"ed “questions considered by the security council at its 838th meeting on 7 august 1958”, noting the charter aim that states should practise tolerance and live together in peace with one another as good neigh"
5,1237 (es-iii),1,pact,415,419,"ether in peace with one another as good neighbours, noting that the arab states have agreed, in the pact of the league of arab states, to strengthen the close relations and numerous ties which link the ar"
6,1237 (es-iii),1,pact,991,995,lcomes the renewed assurances given by the arab states to observe the provision of article 8 of the pact of the league of arab states that each tnember s a‘e shall respect the systems of government establ
7,1237 (es-iii),1,charter,1793,1800,"eneral to make forthwith, in consultation with the governments concerned and in accordance with the charter, and having in mind section i of this resolution, such practical arrangements as would adequately h"
8,1237 (es-iii),1,charter,1952,1959,"uch practical arrangements as would adequately help in upholding the purposes and principles of the charter in relation to lebanon and jordan in the present circumstances, and thereby facilitate the early wi"
9,2252 (es-v),1,treaty,1657,1663,"embly on 26 june 1967;* 5 united nations, treaty series, vol. 75 (1950), no. 972. 6 united nations, treaty sertes, vol. 75 (1950), nos. 970-973. 7 see official records of the general assembly, fifth emer-ge"


In [187]:
cites

,res_id2,res_part,document,id,start,end,context
0,1004 (es-ii),1,charter of the united nations,Charter of the United Nations;26 June 1945,467,496,"and that the general principle of these rights and this freedom is «./firmed for all peoples in the charter of the united nations, convinced that recent events in hungary manifest clearly the desire of the hungarian people to exe"
1,1004 (es-ii),1,charter of the united nations,Charter of the United Nations;26 June 1945,2984,3013,ds to bring an end to the foreign intervention in hungary in accordance with the prin-ciples of the charter of the united nations;
2,1005 (es-ii),1,charter of the united nations,Charter of the United Nations;26 June 1945,903,932,"ering that the repression undertaken by the soviet forces in hungary constitutes a violation of the charter of the united nations and of the peace treaty between hungary and the allied and associated powers, considering that the"
3,1007 (es-ii),1,charter of the united nations,Charter of the United Nations;26 June 1945,330,359,"ust effectively through the international eoooperition stipulated in article 1, paragraph 3, of the charter of the united nations, 1. resolves to undertake on a large scale tmmediate aid for the affected territories by furnishing"
4,1474 (es-iv),1,charter of the united nations,Charter of the United Nations;26 June 1945,2509,2538,"of the republic of the congo; (b) all member states, in accordance with ar-ticles 25 and 49 of the charter of the united nations, to accept and carry out the decisions of the security council and to afford mutual assistance in c"
5,2252 (es-v),1,geneva convention relative to the treatment of prisoners of war,https://www.ohchr.org/en/instruments-mechanisms/instruments/geneva-convention-relative-treatment-prisoners-war,649,712,ld be respected even during the vicissitudes of war; (c) considered that all the obligations of the geneva convention relative to the treatment of prisoners of war of 12 august 1949° should be com-plied with by the parties involved in the conflict; (d) called upo
6,es-6/2,1,charter of the united nations,Charter of the United Nations;26 June 1945,870,899,"dependence of any state or in any other manner inconsistent with the purposes and principles of the charter of the united nations, recognizing the urgent need for immediate termina-tion of foreign armed intervention in afghanista"
7,es-6/2,1,charter of the united nations,Charter of the United Nations;26 June 1945,1511,1540,"ternational law concerning friendly rela-tions and co-operation among states in accordance with the charter of the united nations, expressing its deep concern at the dangerous escala-tion of tension, intensification of rivalry an"
8,es-6/2,1,charter of the united nations,Charter of the United Nations;26 June 1945,2186,2215,"ter-ritorial integrity and political independence of every state is a fundamental principle of the charter of the united nations, any violation of which on any pretext whatsoever is contrary to its aims and purposes; 2. strongly"
9,es-7/2,1,charter of the united nations,Charter of the United Nations;26 June 1945,1386,1415,"rehensive, just and lasting peace in the middle east cannot be es-tablished, in accordance with the charter of the united nations and the relevant united nations resolutions, without the withdrawal of israel from all the occupied"


In [188]:
pd.reset_option("display.max_colwidth")
pd.reset_option("display.max_rows")

"""
1. create a df with the unique titles (cleaned_name column)
2. scan them through the resolution['content]
3. save all in a cites dataframe
4. now the non unique titles (cleaned_name column)
5. scan them non alone through the resolution['content]: they must be accompagned with 'of df['year'], ' (df['year'])', 'df['year'] df['cleaned_name']', '(df['year']) df['cleaned_name']'
6. save everything in cites dataframe
Note: the cites dataframe is resolutions[res_id2], treaty and id, which is composed of the treaty + comma + year
"""


"\n1. create a df with the unique titles (cleaned_name column)\n2. scan them through the resolution['content]\n3. save all in a cites dataframe\n4. now the non unique titles (cleaned_name column)\n5. scan them non alone through the resolution['content]: they must be accompagned with 'of df['year'], ' (df['year'])', 'df['year'] df['cleaned_name']', '(df['year']) df['cleaned_name']'\n6. save everything in cites dataframe\nNote: the cites dataframe is resolutions[res_id2], treaty and id, which is composed of the treaty + comma + year\n"

In [189]:
cites.to_csv("treaty_citations-first-draft.csv", index=False)

In [190]:
"""


Geneva convention. there are many, the first is just geneva convention but the other ones are named by second, third, forth geneva convention. The problem is that the cases where the forth is references, sometimes they dont inlcude the ordinal, so the system detects as the first, but the text next inlcude the date, including 1949
Solution, the geneva convention should also detects a date
Geneva Convention relative to the Protection of Civilian Persons in Time of War, of 12 August 1949 = fourth geneva convention


"""

'\n\n\nGeneva convention. there are many, the first is just geneva convention but the other ones are named by second, third, forth geneva convention. The problem is that the cases where the forth is references, sometimes they dont inlcude the ordinal, so the system detects as the first, but the text next inlcude the date, including 1949\nSolution, the geneva convention should also detects a date\nGeneva Convention relative to the Protection of Civilian Persons in Time of War, of 12 August 1949 = fourth geneva convention\n\n\n'

In [191]:
# group to see the most cited treaties
cites = (
    cites.groupby('id')
         .agg(count=('res_id2', 'count'))
         .reset_index()
)

In [192]:
cites.sort_values(by='count', ascending=False, inplace=True)
print(cites.head(10))

                                                  id  count
0         Charter of the United Nations;26 June 1945     33
2                      Fourth Geneva Convention;1949     26
3  Geneva Convention relative to the Protection o...     22
1  Convention on the Rights of the Child;20 Novem...      1
4  International Covenant on Civil and Political ...      1
5  Rome Statute of the International Criminal Cou...      1
6                        United Nations Charter;1945      1
7  https://www.ohchr.org/en/instruments-mechanism...      1
